# Adaptive Steane SE: circuits → Guppy/HUGR → LER

A compact example for **[[32,20,4]]**. Run the cells in order; optimization, fault analysis and compilation take a few minutes.

Setup (Python 3.12+, from the repository root): `python -m pip install -e ".[notebooks,guppy]"`. Select that environment as the notebook kernel.

After updating SpiderWarp or Guppy, **restart the kernel and run all cells** to load the current code. The setup cell prints the Python environment and Guppy version.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

import stim
from hugr.package import Package
from IPython.display import Markdown, display

from spiderwarp.csscode import CSSCode
from spiderwarp.utils import get_project_root
from spiderwarp.optimized_steane_se import optimized_steane_SE_orchestrator
from spiderwarp.se_protocol_simulation import estimate_protocol_ler

print(f"Python: {sys.executable}")
print(f"Guppy: {version('guppylang')}")

Python: /opt/homebrew/Caskroom/miniforge/base/envs/error-correction/bin/python
Guppy: 0.21.6


**Specify the code and circuit distances.** `code.d` declares the code distance; the dictionary keys declare each supplied gadget's circuit distance. All input gadgets extract **X stabilizers**. The builder constructs their Z duals and derives the primary distance and fault budgets from `code.d`.

For another code, load its matrices and distance through `CSSCode`, and supply its distance-labelled circuits in the same form. The current builder supports identical `H_x`/`H_z` and code distances 2–5.

In [2]:
code_id = "32_20_4"
code = CSSCode.load_code("misc", code_id)  # Includes n, k, d, H_x/H_z and L_x/L_z.
assets = get_project_root() / "assets/circuits/SECircuits"
circuits = {
    d: stim.Circuit.from_file(assets / f"{code_id}_d{d}.stim")
    for d in (1, 2, 4)
}
print(f"Code [[{code.n},{code.k},{code.d}]]; circuit distances: {list(circuits)}")

Code [[32,20,4]]; circuit distances: [1, 2, 4]


**Optimize and export.** The clean cycle runs the maximum-distance Z primary, corrects, then runs the maximum-distance X primary and corrects. A raised primary flag discards that primary syndrome and selects a dual-basis recovery; where required, an additional fresh extraction resolves ambiguity. Raised recovery flags or ambiguous outcomes reject the shot.

Verification is **off by default**. Fault analysis still constructs the branch decisions. To check the resulting policy separately, call `protocol.verify()` (equivalently `verify_se_protocol(protocol)`); or pass `verify=True` during construction. The optional check uses Gottesman's conditions with separate CSS X/Z error weights. The standalone `.py` and `.hugr` contain the control flow and corrections, with no decoder sidecar.

In [3]:
output_dir = get_project_root().parent / "build" / f"{code_id}_protocol_demo"
protocol = optimized_steane_SE_orchestrator(code, circuits, output_dir=output_dir)
# Optional: report = protocol.verify()
print(f"Verification performed: {protocol.verification.verified}; peak SE qubits: {protocol.required_qubits}")
print(f"Correctable faults: {protocol.verification.correctable_faults}; "
      f"analyzed fault order: {protocol.verification.detectable_faults}")
print(f"Guppy: {protocol.source_path}")
print(f"HUGR:  {protocol.package_path}")

Verification performed: False; peak SE qubits: 60
Correctable faults: 1; analyzed fault order: 2
Guppy: /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/steane_se_protocol.py
HUGR:  /Users/boldi/PycharmProjects/AutomatedSteane/build/32_20_4_protocol_demo/steane_se_protocol.hugr


**Load the exported HUGR.** This is a composable function package. To compose the Guppy source with a larger program, import its `se_cycle(data) -> bool` and honor the acceptance bit; `postselected_se_cycle(data)` instead terminates rejected shots. The generated module provides `compile_se_cycle()`: it selects minimal optimization when supported, or direct compilation with Guppy 0.21.6.

In [4]:
loaded_hugr = Package.from_bytes(Path(protocol.package_path).read_bytes())
guppy_cycle = protocol.guppy_function  # Callable from other @guppy functions.
print(f"Reloaded HUGR package with {len(loaded_hugr.modules)} module(s).")

Reloaded HUGR package with 1 module(s).


**Estimate one-cycle block LER.** Choose `prepared_state = "0"` or `"+"` below. All 20 logical qubits start in that state. Logical 0 inputs test logical X/Y errors; logical + inputs test logical Z/Y errors. A shot fails if any logical qubit has the wrong readout after ideal final `LutDecoder` correction.

This runs the adaptive **Stim reference interpreter** for the exported policy. Input preparation and final decoding are ideal. Each executed extraction and correction block is made noisy by `noise_model.py`'s `apply_noise`, using its **default hardware model**, including scheduled memory noise. Classical branch latency and later HUGR backend transformations are not modeled. To override the default, pass `noise_model=CustomErrorModel(...)` from `spiderwarp.noise_model`.

**LER is conditional on acceptance**; discards and a 95% Wilson interval are reported separately. Verification is optional for sampling too.

In [5]:
prepared_state = "0"  # Choose "0" or "+".
shots = 100_000
seed = 20261015
estimate = estimate_protocol_ler(protocol, prepared_state=prepared_state, shots=shots, seed=seed)
print(f"Prepared logical state: {prepared_state}; Stim {stim.__version__}")
print("Default noise parameters:", vars(estimate.noise_model))
lo, hi = estimate.ler_interval_95
rows = ["| Accepted / shots | Discard rate | Logical failures | Block LER | 95% interval |",
        "|---:|---:|---:|---:|---:|",
        f"| {estimate.accepted} / {estimate.shots} | {1 - estimate.acceptance_rate:.2%} | "
        f"{estimate.logical_errors} | {estimate.ler:.3%} | [{lo:.3%}, {hi:.3%}] |"]
display(Markdown("\n".join(rows)))

Prepared logical state: 0; Stim 1.15.0
Default noise parameters: {'p_1q': 2.5e-05, 'p_2q': 0.0008, 'p_meas': 1e-06, 'p_init': 0.0008, 'p_linear_mem': 0.00018899999999999999, 'p_coherent_mem': 0, 'p_leakage': 0.0}


| Accepted / shots | Discard rate | Logical failures | Block LER | 95% interval |
|---:|---:|---:|---:|---:|
| 76197 / 100000 | 23.80% | 1086 | 1.425% | [1.344%, 1.512%] |

Zero observed failures give an upper uncertainty limit, not proof that LER is zero. This estimates one SE cycle from clean logical 0/+ input, conditional on acceptance. Sampling includes all fault orders; optional exact verification uses the bounded circuit-level Pauli model described in its report. Seeds are reproducible with a fixed Stim version and machine architecture.